In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [2]:
df=pd.read_csv('loan_data.csv.xls')

In [3]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [4]:
x = df.drop(columns='loan_status')
y=df.loan_status

In [5]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [6]:
num_col=x.select_dtypes(include='number').columns
obj_cols=x.select_dtypes(include='object').columns

C:\Users\USER\AppData\Local\Temp\ipykernel_3044\1675272409.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols=x.select_dtypes(include='object').columns


In [7]:
x[obj_cols].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [8]:
x['person_education'].unique()

<ArrowStringArray>
['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']
Length: 5, dtype: str

In [ ]:
obj_cols = xtrain.select_dtypes(include="object").columns

C:\Users\USER\AppData\Local\Temp\ipykernel_3044\1324787528.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = xtrain.select_dtypes(include="object").columns


In [14]:
order = ['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']

In [15]:
preprocessing = ColumnTransformer(
    transformers=[
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore"),
            obj_cols.drop("person_education")
        ),
        (
            "ordinal",
            OrdinalEncoder(categories=[order]),
            ["person_education"]
        )
    ],
    remainder="passthrough"
)

## SVM_CLASSIFIER ##


In [17]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
svc_model = Pipeline(steps=[
    ("preprocessing", preprocessing),
    ("model", SVC(kernel='rbf', C=1.0))
])

svc_model.fit(xtrain, ytrain)

y_pred = svc_model.predict(xtest)

print("Accuracy:", accuracy_score(ytest, y_pred))
print(classification_report(ytest, y_pred))

Accuracy: 0.8017777777777778
              precision    recall  f1-score   support

           0       0.80      0.99      0.89      6990
           1       0.81      0.15      0.25      2010

    accuracy                           0.80      9000
   macro avg       0.81      0.57      0.57      9000
weighted avg       0.80      0.80      0.74      9000



## SVM_REGRESSOR

In [18]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

svr_model = Pipeline(steps=[
    ("preprocessing", preprocessing),
    ("model", SVR(kernel='rbf', C=1.0))
])

svr_model.fit(xtrain, ytrain)

y_pred = svr_model.predict(xtest)

print("R2 Score:", r2_score(ytest, y_pred))
print("MSE:", mean_squared_error(ytest, y_pred))

R2 Score: 0.12440102229652183
MSE: 0.1518775071214333


## SVM_Random Forest Classifier

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = Pipeline(steps=[
    ("preprocessing", preprocessing),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

rf_model.fit(xtrain, ytrain)

y_pred = rf_model.predict(xtest)

print("Accuracy:", accuracy_score(ytest, y_pred))
print(classification_report(ytest, y_pred))

Accuracy: 0.9284444444444444
              precision    recall  f1-score   support

           0       0.94      0.97      0.95      6990
           1       0.89      0.78      0.83      2010

    accuracy                           0.93      9000
   macro avg       0.91      0.87      0.89      9000
weighted avg       0.93      0.93      0.93      9000



## SVM_Logistic Regression ##

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr_model = Pipeline(steps=[
    ("preprocessing", preprocessing),
    ("model", LogisticRegression(max_iter=1000))
])

lr_model.fit(xtrain, ytrain)

y_pred = lr_model.predict(xtest)

print("Accuracy:", accuracy_score(ytest, y_pred))
print(classification_report(ytest, y_pred))

Accuracy: 0.8853333333333333
              precision    recall  f1-score   support

           0       0.92      0.93      0.93      6990
           1       0.75      0.73      0.74      2010

    accuracy                           0.89      9000
   macro avg       0.84      0.83      0.83      9000
weighted avg       0.88      0.89      0.88      9000



c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
